In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict
from qiskit_metal.toolbox_python.attr_dict import Dict
from qiskit_metal.qlibrary.core import QComponent

print(f"Qiskit-Metal version: {metal.__version__}")

Qiskit-Metal version: 0.5.3.post1


In [2]:
from qiskit_metal.qlibrary.user_components.circmong_singleJJ import CircmonG
print("CircmonG imported successfully.")
print()
print("Junction key in HFSS: 'rect_jj1'")
print("For component name 'TC1':")
print("  rect : JJ_rect_Lj_TC1_rect_jj1")
print("  line : JJ_Lj_TC1_rect_jj1_")

CircmonG imported successfully.

Junction key in HFSS: 'rect_jj1'
For component name 'TC1':
  rect : JJ_rect_Lj_TC1_rect_jj1
  line : JJ_Lj_TC1_rect_jj1_


In [3]:
# ── Create design ─────────────────────────────────────────────────────────
# Set chip size to match W (the metallic plane side).
# A small border is added so the HFSS airbox does not clip the component.
W_um = 800    # must match the W option below
H_um = 280 # substrate height

design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = f'{W_um}um'
design.chips.main.size['size_y'] = f'{W_um}um'
design.chips.main.size['size_z'] = f'{-H_um}um'
design.chips.main.size['center_x'] = '0um'
design.chips.main.size['center_y'] = '0um'

gui = MetalGUI(design)
print("Design created. MetalGUI opened.")

Design created. MetalGUI opened.


In [4]:
from qiskit_metal.qlibrary.user_components.circmong_singleJJ import CircmonG

circmon1 = CircmonG(design, 'TC1', options=dict(
    pos_x      = '0mm',
    pos_y      = '0mm',
    pad_r      = '150um',   # qubit island radius — primary Ec knob
    pad2pk_gap = '70um',   # slot width          — secondary Ec knob
    jj_width   = '20um',
    jj_angle   = 0,         # 0 deg = east (+x); single JJ only
    jj_in_pad  = '2um',
    jj_in_pk   = '2um',
))

In [5]:
design.overwrite_enabled = True
gui.rebuild()
gui.autoscale()

print("CircmonG built.")
print()
print(f"  pad_r      : {circmon1.options.pad_r}")
print(f"  pad2pk_gap : {circmon1.options.pad2pk_gap}")
print(f"  jj_width   : {circmon1.options.jj_width}")
print(f"  jj_angle   : {circmon1.options.jj_angle} deg")
print()
print("Junction table:")
print(design.qgeometry.tables['junction'][
    ['component','name','width','hfss_inductance','hfss_capacitance']])

CircmonG built.

  pad_r      : 150um
  pad2pk_gap : 70um
  jj_width   : 20um
  jj_angle   : 0 deg

Junction table:
  component      name  width hfss_inductance hfss_capacitance
0         1  rect_jj1   0.02              Lj               Cj


In [6]:
# This cell defines the hfss sim setup

from qiskit_metal.analyses.quantization import EPRanalysis

# ── Create EPRanalysis object ─────────────────────────────────────────────
eig_qb = EPRanalysis(design, "hfss")

# ── Simulation setup ──────────────────────────────────────────────────────
eig_qb.sim.setup.name          = 'Qbit_Setup'
eig_qb.sim.setup.n_modes       = 1
eig_qb.sim.setup.max_passes    = 15
eig_qb.sim.setup.max_delta_f   = 0.1    # convergence: 0.1% change in freq
eig_qb.sim.setup.min_freq_ghz  = 1.0
eig_qb.sim.setup.min_converged = 2

# ── Junction variable (single JJ) ─────────────────────────────────────────
# Single JJ: Ej = (Phi0/2pi)^2 / Lj  (no parallel combination factor)
# For f_01 ~ 3.5 GHz at Ec ~ 140 MHz: Ej ~ 11.8 GHz -> Lj ~ 13.5 nH
# Adjust Lj to tune f_01.
eig_qb.sim.setup.vars = Dict(
    Lj = '13 nH',   # single JJ inductance — adjust to tune f_01
    Cj = '2 fF',    # shunt capacitance
)

# ── HFSS box buffer ───────────────────────────────────────────────────────
eig_qb.sim.renderer.options['x_buffer_width_mm'] = 0.5
eig_qb.sim.renderer.options['y_buffer_width_mm'] = 0.5

print("EPRanalysis setup:")
eig_qb.sim.setup

EPRanalysis setup:


{'name': 'Qbit_Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1.0,
 'n_modes': 1,
 'max_delta_f': 0.1,
 'max_passes': 15,
 'min_passes': 1,
 'min_converged': 2,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': '13 nH', 'Cj': '2 fF'}}

In [7]:
# ── Render design to HFSS ─────────────────────────────────────────────────
# Prerequisite: Ansys HFSS must be running with a valid licence.

# Get the renderer from the sim object
hfss = eig_qb.sim.renderer

#Connect to the running HFSS instance
hfss.start()

#Create new hfss design
hfss.new_ansys_design('Qbit_hfss', 'eigenmode')

# Pull hfss variables directly from  already-defined setup.vars
for var, val in eig_qb.sim.setup.vars.items():
    hfss.pinfo.design.set_variable(var, val)

# to reuse an existing hfss design use the command below
#hfss.activate_ansys_design('_design_name_')

# Render only — no analysis launched
hfss.render_design(
    selection=['TC1'],
    open_pins=[],
    box_plus_buffer=True
)

eig_qb.sim.print_run_args()
print()
print("HFSS rendering complete.")
print()

# ── Confirm JJ object names that pyEPR will look for ─────────────────────
comp = 'TC1'
print(f"Expected HFSS JJ object names for component '{comp}':")
print(f"  rect : JJ_rect_Lj_{comp}_rect_jj1")
print(f"  line : JJ_Lj_{comp}_rect_jj1_")

INFO 04:44PM [connect_project]: Connecting to Ansys Desktop API...
INFO 04:44PM [load_ansys_project]: 	Opened Ansys App
INFO 04:44PM [load_ansys_project]: 	Opened Ansys Desktop v2024.2.0
INFO 04:44PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/Kobi/Documents/Ansoft/
	Project:   Project37
INFO 04:44PM [connect_design]: No active design found (or error getting active design).
INFO 04:44PM [connect]: 	 Connected to project "Project37". No design detected
INFO 04:44PM [connect_design]: 	Opened active design
	Design:    Qbit_hfss [Solution type: Eigenmode]
WARNING 04:44PM [connect_setup]: 	No design setup detected.
WARNING 04:44PM [connect_setup]: 	Creating eigenmode default setup.
INFO 04:44PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


This analysis object run with the following kwargs:
{}


HFSS rendering complete.

Expected HFSS JJ object names for component 'TC1':
  rect : JJ_rect_Lj_TC1_rect_jj1
  line : JJ_Lj_TC1_rect_jj1_


In [8]:
# create spr boxes for later field calculations
from qiskit_metal.qlibrary.user_components.spr_box_create3 import *
spr_boxes = create_circmong_spr_boxes(design, circmon1)


  [OK] MA_ground_TC1                 XY=4x(150um+70um)  t=5um


In [9]:
# Run hfss eigenmode analysis
eig_qb.sim.renderer.pinfo.setup.analyze()
_solved = True
print("  HFSS solve complete via analyze_setup().")

INFO 04:45PM [analyze]: Analyzing setup Setup


  HFSS solve complete via analyze_setup().


In [10]:
# print eignenmodes
try:
    freqs = eig_qb.get_frequencies()
    print("Eigenmode frequencies (linearised with Lj):")
    print(freqs)
    print()
    try:
        print("Convergence per pass:")
        print(eig_qb.sim.convergence_f)
    except Exception:
        pass
except Exception as e:
    print(f"  [get_frequencies: {e}]")
    print("  Ensure the HFSS solve ran successfully before proceeding to Step 5.")

Design "Qbit_hfss" info:
	# eigenmodes    1
	# variations    1
Design "Qbit_hfss" info:
	# eigenmodes    1
	# variations    1
Eigenmode frequencies (linearised with Lj):
                Freq. (GHz)  Quality Factor
variation mode                             
0         0        4.377964             inf

Convergence per pass:
{}


In [11]:
# ── Configure the single junction for EPR ────────────────────────────────
# Component name must match what was used in CircmonG(design, 'TC1', ...)
comp = 'TC1'
eig_qb.setup.junctions = Dict()  # clear any stale/defaults unction entries

# Register the single JJ  (key used in make_jj() was 'rect_jj1')
# HFSS object names follow the convention:
#   rect : JJ_rect_Lj_{comp}_{key}
#   line : JJ_Lj_{comp}_{key}_
eig_qb.setup.junctions['rect_jj1'] = Dict(
    rect        = f'JJ_rect_Lj_{comp}_rect_jj1',
    line        = f'JJ_Lj_{comp}_rect_jj1_',
    Lj_variable = 'Lj',
    Cj_variable = 'Cj',
)
eig_qb.setup.junctions

{'rect_jj1': {'rect': 'JJ_rect_Lj_TC1_rect_jj1',
  'line': 'JJ_Lj_TC1_rect_jj1_',
  'Lj_variable': 'Lj',
  'Cj_variable': 'Cj'}}

In [12]:
# register the box for SPR analysis
eig_qb.setup.dissipatives = Dict(
    dielectrics_bulk    = ['main', 'MA_ground_TC1'],  # ← solid volume integral
    dielectric_surfaces = [],
    resistive_surfaces  = [],
    seams               = [],
)

In [ ]:
# ── Run EPR analysis ──────────────────────────────────────────────────────
# run_epr() integrates E-field over the JJ rectangle and computes
# the inductive participation ratio p_mj and participation sign s_mj.
print("Running EPR field integration...")
eig_qb.run_epr()

In [ ]:
# Full results table — includes p_diel per object, frequencies, anharmonicity
# ── Step 1: field energy integrals ────────────────────────────────────────
#eig_qb.get_stored_energy()

# ── Step 2: quantum analysis — capture the return value ───────────────────
#epra = eig_qb.run_analysis()
#print([m for m in dir(epra) if 'diel' in m.lower() or 'chi' in m.lower() or 'p_' in m.lower()])

In [13]:

import pyEPR as epr
import pathlib

da = eig_qb.sim.renderer.epr_distributed_analysis

# ── Locate the actual .npz data file on disk ─────────────────────────────
data_dir = pathlib.Path(da.data_filename).parent

npz_candidates = [
    f for f in data_dir.glob('*.npz')
    if 'HamiltonianResultsContainer' not in f.name
]

if not npz_candidates:
    raise FileNotFoundError(
        f"No pyEPR data .npz found in: {data_dir}\n"
        f"Expected by da.data_filename: {da.data_filename}"
    )

# Pick the most recently modified one (handles multiple runs in same folder)
actual_data_file = max(npz_candidates, key=lambda f: f.stat().st_mtime)

print(f"da.data_filename (may be stale) : {da.data_filename}")
print(f"Actual .npz found on disk       : {actual_data_file}")
print()

# ── Instantiate QuantumAnalysis from the real file ───────────────────────
qa = epr.QuantumAnalysis(str(actual_data_file))

# ── Run full quantum analysis ─────────────────────────────────────────────
print("Running analyze_all_variations()...")
print("  cos_trunc  = 8  — Taylor expansion of cos(phi) to 8th order")
print("  fock_trunc = 7  — Fock space size: levels |0> to |6>")
print()

qa.analyze_all_variations(
    cos_trunc  = 8,
    fock_trunc = 7,
)

print("Quantum analysis complete.")



WARNING 04:47PM [__init__]: <p>Error: <class 'IndexError'></p>


da.data_filename (may be stale) : C:\data-pyEPR\Project37\Qbit_hfss\2026-04-07 16-46-44.npz
Actual .npz found on disk       : C:\data-pyEPR\Project37\Qbit_hfss\2026-04-07 16-27-38.npz

	 Differences in variations:


Running analyze_all_variations()...
  cos_trunc  = 8  — Taylor expansion of cos(phi) to 8th order
  fock_trunc = 7  — Fock space size: levels |0> to |6>


 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
Variation 0

Starting the diagonalization
Finished the diagonalization
Pm_norm=
modes
0    1.000503
dtype: float64

Pm_norm idx =
   rect_jj1
0      True
*** P (participation matrix, not normlz.)
   rect_jj1
0  0.991684

*** S (sign-bit matrix)
   s_rect_jj1
0           1
*** P (participation matrix, normalized.)
      0.99

*** Chi matrix O1 PT (MHz)
    Diag is anharmonicity, off diag is full cross-Kerr.
       188

*** Chi matrix ND (MHz) 
    206+0j

*** Frequencies O1 PT (MHz)
0    4190.393359
dtype: float64

*** Frequencies ND (MHz)
0 

In [16]:
import pandas as pd, numpy as np

var = '0'

# ── Chi matrix and anharmonicity (correct method: get_chis) ───────────────
chis  = qa.get_chis(variation=var)                    # DataFrame, MHz
alpha = float(abs(chis.values[0, 0].real))            # diagonal = anharmonicity

# ── EC and EJ ─────────────────────────────────────────────────────────────
Lj_nH = float(eig_qb.sim.setup.vars.Lj.split()[0])
EC    = alpha                                          # MHz  (transmon: EC ≈ α)
EJ    = (2.07e-15/(2*np.pi))**2 / (Lj_nH*1e-9) / (6.626e-34*1e6)   # MHz
EJ_EC = EJ / EC

# ── p_MA (correct method: calc_p_electric_volume on da) ───────────────────
p_MA = da.calc_p_electric_volume(obj_name='MA_ground_TC1', variation=var)

# ── Summary table ──────────────────────────────────────────────────────────
pd.DataFrame(
    [[f'{EJ_EC:.0f}', f'{EC:.1f}', f'{alpha:.1f}', f'{p_MA:.4f}']],
    columns=['EJ/EC', 'EC (MHz)', 'α ND (MHz)', 'p_MA']
)

TypeError: QuantumAnalysis.get_chis() got an unexpected keyword argument 'variation'